# skforecast-ai — Demo Phase 6

This notebook demonstrates **Phase 6: LLM Provider Abstraction**.

The `llm/provider` module parses provider strings (e.g. `"openai:gpt-4o-mini"`,
`"ollama:qwen2.5:7b-instruct"`) and creates the appropriate Pydantic AI model
instances. No agent logic yet — only model instantiation and validation.

## Setup

In [1]:
import skforecast_ai
from skforecast_ai.llm.provider import (
    SUPPORTED_PROVIDERS,
    create_model,
    parse_model_string,
)

print(f"skforecast-ai version: {skforecast_ai.__version__}")
print(f"Supported providers: {sorted(SUPPORTED_PROVIDERS)}")

skforecast-ai version: 0.1.0
Supported providers: ['anthropic', 'google', 'groq', 'mistral', 'ollama', 'openai']


## 1. Parsing provider strings

`parse_model_string()` splits a provider string into `(provider, model_name)`.
It handles Ollama's double-colon format (e.g. `qwen2.5:7b-instruct`) by splitting
only on the first colon.

In [2]:
# Standard cloud providers
print(parse_model_string("openai:gpt-4o-mini"))
print(parse_model_string("anthropic:claude-sonnet-4-5"))
print(parse_model_string("google:gemini-2.0-flash"))
print(parse_model_string("groq:llama-3.3-70b-versatile"))
print(parse_model_string("mistral:mistral-large-latest"))

('openai', 'gpt-4o-mini')
('anthropic', 'claude-sonnet-4-5')
('google', 'gemini-2.0-flash')
('groq', 'llama-3.3-70b-versatile')
('mistral', 'mistral-large-latest')


In [3]:
# Ollama with model tag (double colon)
print(parse_model_string("ollama:qwen2.5:7b-instruct"))
print(parse_model_string("ollama:llama3.1:8b"))

('ollama', 'qwen2.5:7b-instruct')
('ollama', 'llama3.1:8b')


In [4]:
# None → Tier 0 (no LLM)
print(parse_model_string(None))

(None, None)


## 2. Error handling — invalid strings

Clear `ValueError` messages guide the user when the format is wrong.

In [5]:
# Missing provider prefix
try:
    parse_model_string("gpt-4o-mini")
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: Invalid LLM string 'gpt-4o-mini'. Expected format 'provider:model_name' (e.g. 'openai:gpt-4o-mini', 'ollama:qwen2.5:7b-instruct').


In [6]:
# Unsupported provider
try:
    parse_model_string("foobar:some-model")
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: Unsupported provider 'foobar'. Supported providers: ['anthropic', 'google', 'groq', 'mistral', 'ollama', 'openai'].


## 3. Creating models with `create_model()`

- **Cloud providers** → returns the raw string (Pydantic AI resolves natively)
- **Ollama** → returns an `OpenAIChatModel` with custom base URL
- **None** → returns `None` (Tier 0 deterministic mode)

In [7]:
# Tier 0: no LLM
model = create_model(None)
print(f"create_model(None) → {model}")
print(f"Type: {type(model)}")

create_model(None) → None
Type: <class 'NoneType'>


In [8]:
# Cloud provider: string passthrough
model = create_model("openai:gpt-4o-mini")
print(f"create_model('openai:gpt-4o-mini') → {model!r}")
print(f"Type: {type(model)}")

create_model('openai:gpt-4o-mini') → 'openai:gpt-4o-mini'
Type: <class 'str'>


In [9]:
# Ollama: explicit OpenAIChatModel instantiation
model = create_model("ollama:qwen2.5:7b-instruct")
print(f"create_model('ollama:qwen2.5:7b-instruct') → {model!r}")
print(f"Type: {type(model).__name__}")
print(f"Model name: {model.model_name}")

create_model('ollama:qwen2.5:7b-instruct') → OllamaModel()
Type: OllamaModel
Model name: qwen2.5:7b-instruct


In [10]:
# Ollama with custom base URL (e.g. remote server)
model = create_model(
    "ollama:qwen2.5:14b-instruct",
    base_url="http://192.168.1.50:11434/v1",
)
print(f"Model name: {model.model_name}")
print(f"Type: {type(model).__name__}")

Model name: qwen2.5:14b-instruct
Type: OllamaModel


## 4. Connectivity check

`check_ollama_reachable()` verifies an Ollama instance is responding.
It raises a `ConnectionError` with an actionable message if not.

In [11]:
from skforecast_ai.llm.provider import check_ollama_reachable

try:
    reachable = check_ollama_reachable()
    print(f"Ollama reachable at default URL: {reachable}")
except ConnectionError as e:
    print(f"ConnectionError: {e}")

Ollama reachable at default URL: True


In [12]:
from skforecast_ai.llm.provider import check_ollama_reachable, create_model

check_ollama_reachable()
model = create_model("ollama:qwen3:8b")

## 5. Tier 0 safety — no pydantic-ai required for basic usage

The `parse_model_string` function and cloud provider path in `create_model`
never import `pydantic_ai`. Only the Ollama path triggers the lazy import.
This means Tier 0 (`llm=None`) works without the `[llm]` extra installed.

In [13]:
# These work without pydantic-ai installed
assert create_model(None) is None
assert create_model("openai:gpt-4o-mini") == "openai:gpt-4o-mini"
assert parse_model_string("anthropic:claude-sonnet-4-5") == ("anthropic", "claude-sonnet-4-5")

print("All Tier 0 operations work without pydantic-ai.")

All Tier 0 operations work without pydantic-ai.


## Summary

Phase 6 provides the **LLM provider abstraction layer**:

- `parse_model_string()` — validates and splits provider strings
- `create_model()` — resolves to the correct Pydantic AI model
- `check_ollama_reachable()` — connectivity validation

Cloud providers use **string passthrough** (less coupling to pydantic-ai internals).
Only Ollama gets explicit `OpenAIChatModel` instantiation (needs custom base URL + API key).
All pydantic-ai imports are **lazy** — Tier 0 never touches them.